# A3 — Adaptive Content and Cold-Start Readiness Audit

This CPU-only audit decides which retrieval branch should be built next. It measures whether current item metadata can distinguish cold videos, whether cold targets belong to known authors, and whether users interacted with those authors before each query.

No model is trained and no checkpoint is downloaded. Use a High-RAM CPU runtime when available.

In [ ]:
from google.colab import drive, files, userdata
drive.mount('/content/drive')

from getpass import getpass
from pathlib import Path
import hashlib, importlib.util, json, shutil, subprocess, sys, zipfile

DRIVE_ROOT = Path('/content/drive/MyDrive/recsys')
ADAPTIVE_ROOT = DRIVE_ROOT / 'data/gold/adaptive/v1'
BASE_GOLD_ROOT = DRIVE_ROOT / 'data/gold/two_tower/v1_colab'
PROTOCOL_DIR = ADAPTIVE_ROOT / 'evaluation_protocol/v2'
WORK_ROOT = Path('/content/recsys_content_readiness')
SOURCE_ROOT = WORK_ROOT / 'runtime'
OUTPUT_DIR = WORK_ROOT / 'reports/content_readiness'
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('adaptive root:', ADAPTIVE_ROOT)
print('static items:', BASE_GOLD_ROOT / 'item_features/static')

In [ ]:
required = {'pyarrow': 'pyarrow', 'pandas': 'pandas', 'huggingface_hub': 'huggingface_hub>=0.32'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])
import pandas as pd
print('CPU runtime ready')

In [ ]:
required_paths = [
    ADAPTIVE_ROOT / 'sequence_examples/train',
    ADAPTIVE_ROOT / 'sequence_examples/validation',
    PROTOCOL_DIR / 'queries.parquet',
    BASE_GOLD_ROOT / 'item_features/static',
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
assert not missing_paths, 'Missing A3 inputs:\n' + '\n'.join(missing_paths)
print('Drive preflight: PASS')

## Download the small tested runtime

In [ ]:
from huggingface_hub import hf_hub_download

HF_REPO_ID = 'khoa1505/recsys-artifacts'
RUNTIME_EXPECTED_SHA256 = '527981a45786853984bd498b4d5cf7a5138a7e6979ecb0da2d3a97ac6d11e578'
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('HF_TOKEN loaded from Colab Secrets.')
except Exception as exc:
    print('Colab Secrets unavailable:', type(exc).__name__)
    HF_TOKEN = getpass('Paste Hugging Face READ token: ')
assert HF_TOKEN and HF_TOKEN.startswith('hf_')
runtime_zip = Path(hf_hub_download(HF_REPO_ID, 'runtime/adaptive_eval_runtime.zip', repo_type='model', token=HF_TOKEN))

def sha256(path: Path, block_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        while block := stream.read(block_size):
            digest.update(block)
    return digest.hexdigest()

assert sha256(runtime_zip) == RUNTIME_EXPECTED_SHA256, 'Runtime ZIP hash mismatch'
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True)
with zipfile.ZipFile(runtime_zip) as archive:
    assert archive.testzip() is None
    archive.extractall(SOURCE_ROOT)
audit_script = SOURCE_ROOT / 'scripts/audit_adaptive_content_readiness.py'
assert audit_script.exists(), f'Missing audit script: {audit_script}'
print('runtime integrity: PASS')

## Run the audit

The scan is Drive-bound and prints progress for every group of shards. It uses strict-prior events when checking author affinity.

In [ ]:
command = [
    sys.executable, str(audit_script),
    '--adaptive-root', str(ADAPTIVE_ROOT),
    '--base-gold-root', str(BASE_GOLD_ROOT),
    '--protocol-dir', str(PROTOCOL_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--split', 'validation',
]
log_path = Path('/content/adaptive_content_readiness.log')
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
with log_path.open('w') as log_file:
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(log_path.read_text(errors='replace').splitlines()[-160:])
    raise RuntimeError(f'A3 failed with exit code {return_code}. Last log lines:\n{tail}')

## Inspect the model-input decision

In [ ]:
report = json.loads((OUTPUT_DIR / 'content_readiness.json').read_text())
quality = pd.read_csv(OUTPUT_DIR / 'feature_quality.csv')
print(json.dumps({
    'candidate_items': report['candidate_items'],
    'metadata_unique_rate': report['metadata_only_unique_signature_rate'],
    'author_content_unique_rate': report['author_content_unique_signature_rate'],
    'cold_targets': report['cold_targets'],
}, indent=2))
display(quality.sort_values('candidate_missing_rate', ascending=False))

## Save the lightweight report

The output contains aggregate JSON/CSV and a small per-query audit. No model or embedding cache is stored.

In [ ]:
archive_path = Path('/content/adaptive_content_readiness_a3.zip')
if archive_path.exists():
    archive_path.unlink()
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file():
            archive.write(path, Path('content_readiness_a3') / path.relative_to(OUTPUT_DIR))
    if log_path.exists():
        archive.write(log_path, 'logs/adaptive_content_readiness.log')
persistent_dir = DRIVE_ROOT / 'artifacts/adaptive/content_readiness_a3'
persistent_dir.mkdir(parents=True, exist_ok=True)
persistent_path = persistent_dir / archive_path.name
shutil.copy2(archive_path, persistent_path)
print('saved:', persistent_path, archive_path.stat().st_size, 'bytes')
files.download(str(archive_path))